# ❓ NB-15: Question Generation Fine-tuning (Response→Question)
**Goal:** Fine-tune T5 to reverse Claude: given a response, generate the question.

**Modality:** Reverse QA | **Model:** google/flan-t5-base

In [ ]:
!pip install transformers datasets -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset
import torch

MODEL = "google/flan-t5-base"
tok = T5Tokenizer.from_pretrained(MODEL)
model = T5ForConditionalGeneration.from_pretrained(MODEL)

# Source: response → Target: original user question
src = ["generate question: " + d["response"][:400] for d in data]
tgt = [d["user"] for d in data]

def preprocess(batch):
    inp = tok(batch["src"], max_length=256, truncation=True, padding="max_length")
    lab = tok(batch["tgt"], max_length=128, truncation=True, padding="max_length")
    inp["labels"] = [[(t if t != tok.pad_token_id else -100) for t in l] for l in lab["input_ids"]]
    return inp

ds = Dataset.from_dict({"src":src,"tgt":tgt}).map(preprocess, batched=True, remove_columns=["src","tgt"]).train_test_split(0.1)
args = Seq2SeqTrainingArguments("./t5-qgen", num_train_epochs=5,
    per_device_train_batch_size=8, predict_with_generate=True,
    fp16=torch.cuda.is_available(), evaluation_strategy="epoch", report_to="none")
Seq2SeqTrainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
    tokenizer=tok, data_collator=DataCollatorForSeq2Seq(tok, model)).train()
print("✅ Question generator fine-tuned!")


In [ ]:
# Inference: give response, get question
resp = "Neural networks learn by adjusting weights through backpropagation, minimizing loss over training data."
inp = tok("generate question: " + resp, return_tensors="pt").input_ids
out = model.generate(inp, max_new_tokens=64, num_beams=4)
print("Generated Q:", tok.decode(out[0], skip_special_tokens=True))
